# MIMIC-CXR Multimodal Pairing & Directory Flattening

This notebook performs two tasks:
1. **Flattens the subset reports folder**: Moves all report `.txt` files directly into `subset_reports/` and deletes the empty nested subfolders.
2. **Pairs Images and Reports**: Creates a master CSV `1mimic_subset_1000_paired.csv` linking each image path (whether stored in flat format or nested folders) to its corresponding flattened report text path.

In [ ]:
import os
import shutil
import pandas as pd

## 1. Flatten the Reports Directory
We search recursively for all `.txt` report files inside `restricted_Dowloaded_dataset/subset_reports/`, move them to the root of that folder, and delete the empty subfolders.

In [ ]:
reports_root = os.path.join('restricted_Dowloaded_dataset', 'subset_reports')

print("Finding all report files recursively...")
txt_files = []
for root, dirs, files in os.walk(reports_root):
    for file in files:
        if file.endswith('.txt'):
            txt_files.append(os.path.join(root, file))

print(f"Found {len(txt_files)} report files. Moving them to the root of: {reports_root}...")
moved_count = 0
for file_path in txt_files:
    filename = os.path.basename(file_path)
    dest_path = os.path.join(reports_root, filename)
    
    # Move only if the file is currently inside a nested subfolder
    if os.path.dirname(os.path.abspath(file_path)) != os.path.abspath(reports_root):
        shutil.move(file_path, dest_path)
        moved_count += 1

print(f"Successfully moved {moved_count} files.")

print("Cleaning up empty nested directories...")
deleted_dirs = 0
for root, dirs, files in os.walk(reports_root, topdown=False):
    for d in dirs:
        dir_path = os.path.join(root, d)
        if os.path.exists(dir_path) and not os.listdir(dir_path):
            os.rmdir(dir_path)
            deleted_dirs += 1

print(f"Deleted {deleted_dirs} empty subdirectories. Reports folder is now flat!")

## 2. Load the Image Mapping CSV
We load the downloaded image file mapping to build our paired dataset.

In [ ]:
image_csv = '1mimic_subset_1000_images.csv'
df_images = pd.read_csv(image_csv)
print(f"Loaded image mapping with {len(df_images)} rows.")

## 3. Pair Images with the Flat Reports
We check for images and reports in both **flat** and **nested** folders to ensure the script finds all downloaded files, then generates the master CSV.

In [ ]:
paired_rows = []
missing_images = 0
missing_reports = 0

for _, row in df_images.iterrows():
    sub_id = str(int(row['subject_id']))
    std_id = str(int(row['study_id']))
    prefix = sub_id[:2]
    
    # Check image location: either flat in root, or nested in files/
    image_flat = os.path.join('restricted_Dowloaded_dataset', os.path.basename(row['image_path']))
    image_nested = os.path.join('restricted_Dowloaded_dataset', row['image_path'])
    
    if os.path.exists(image_flat):
        image_local = image_flat
    elif os.path.exists(image_nested):
        image_local = image_nested
    else:
        image_local = None
        missing_images += 1
        
    # Check report location: either flat in subset_reports/, or nested in subset_reports/pXX/
    report_flat = os.path.join(reports_root, f"s{std_id}.txt")
    report_nested = os.path.join(reports_root, f"p{prefix}", f"p{sub_id}", f"s{std_id}.txt")
    
    if os.path.exists(report_flat):
        report_local = report_flat
    elif os.path.exists(report_nested):
        report_local = report_nested
    else:
        report_local = None
        missing_reports += 1
        
    if image_local and report_local:
        paired_rows.append({
            'subject_id': row['subject_id'],
            'study_id': row['study_id'],
            'Pneumonia': row['Pneumonia'],
            'No Finding': row['No Finding'],
            'image_path': image_local,
            'report_path': report_local
        })

df_paired = pd.DataFrame(paired_rows)
df_paired.to_csv('1mimic_subset_1000_paired.csv', index=False)

print(f"\nProcess Complete!")
print(f"- Paired successfully: {len(df_paired)} rows")
print(f"- Missing local images: {missing_images}")
print(f"- Missing local reports: {missing_reports}")
print(f"Saved master paired file to: 1mimic_subset_1000_paired.csv")

## 4. Preview the Paired Dataset

In [ ]:
df_paired.head()